In [1]:
import sys
import os
import time
import uuid
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Setup context
from dotenv import load_dotenv
load_dotenv()
from core import enable_logging
enable_logging()
# 将项目根目录加入模块路径
from agent.BasicAgent import BasicAgent
from core.llm import EasyLLM
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill
from skill.yaml_loader import YAMLSkillLoader, MarkdownSkillLoader
from skill.folder_loader import FolderSkillLoader
from skill import MetaSkill
from pydantic import BaseModel,Field
from Tool import Tool
from skill import BaseSkill
from skill import SkillConfig
class TranslateParams(BaseModel):
    text: str = Field(description="要翻译的文本")
    target_lang: str = Field(default="en", description="目标语言")

class TranslateTool(Tool):
    def __init__(self):
        super().__init__("translate_tool", "将文本翻译为目标语言", TranslateParams)

    def run(self, parameters: dict) -> str:
        # 实际翻译逻辑
        return f"Translated: {parameters['text']}"

# 2. 定义 Skill
class TranslateSkill(BaseSkill):
    def __init__(self):
        config = SkillConfig(
            name="translate",
            description="多语言翻译技能",
            version="1.0.0",
            tags=["translate", "language", "i18n"],
            priority=5,
        )
        super().__init__(config)

    def get_tools(self) -> list:
        return [TranslateTool()]

    def get_prompt(self) -> str:
        return """## 翻译能力
你具备多语言翻译能力。当用户要求翻译时，请使用 translate_tool 工具。
- 支持中英日韩等多种语言
- 可以自动识别源语言
"""

def test_invoke_without_tool(agent):

    agent.clear_history()

    result=agent.invoke("你好，请介绍一下你自己")
    print(result)

async def test_ainvoke_without_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke("你好，请介绍一下你自己")
    print(result)

def test_stream_without_tool(agent):
    agent.clear_history()

    agent.stream_invoke("你好，请介绍一下你自己")

async def test_astream_without_tool(agent):
    agent.clear_history()

    await agent.astream_invoke("你好，请介绍一下你自己")

def test_invoke_with_tool(agent):
    agent.clear_history()

    result=agent.invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

async def test_ainvoke_with_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

def test_stream_with_tool(agent):
    agent.clear_history()


    agent.stream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")

async def test_astream_with_tool(agent):
    agent.clear_history()

    await agent.astream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")



In [2]:
llm= EasyLLM(provider="openai",base_url="http://127.0.0.1:5124/v1",api_key="122",model="qwen3.5-9b")

agent=BasicAgent(name="test_skill", llm=llm,reasoning={"effort":"high"},verbose_thinking=True)

2026-04-19 01:57:02,114 | INFO | EasyLLM 初始化完成: provider=openai, model=qwen3.5-9b
2026-04-19 01:57:02,263 | INFO | BasicAgent 'test_skill' 初始化完成，工具调用: 禁用，provider: openai


In [6]:
test_invoke_without_tool(agent)

2026-04-19 01:57:43,297 | INFO | 对话历史已清空
2026-04-19 01:57:43,298 | INFO | 使用普通模式调用智能体
2026-04-19 01:57:59,326 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"




你好。我是你的 AI 助手，专注于协助完成软件工程任务、解答问题及处理工作流。

**核心能力：**
- 代码编写、审查与调试
- 任务规划与执行
- 文档编写与整理

请直接提出你的需求，我会优先推进实际工作。


In [ ]:
agent.get_canonical_history()

In [ ]:
await test_ainvoke_without_tool(agent)

In [ ]:
test_stream_without_tool(agent)

In [7]:
await test_astream_without_tool(agent)

2026-04-19 01:58:11,670 | INFO | 对话历史已清空
2026-04-19 01:58:11,700 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"


thinking content:
Thinking Process:

1.  **Analyze the Request:**
    *   User: "你好，请介绍一下你自己" (Hello, please introduce yourself.)
    *   Intent: The user wants to know who I am, my capabilities, and my role.
    *   Constraints: I need to follow the system instructions provided in the prompt (system interaction rules, task execution principles, risk & safety, tone & style, output efficiency).

2.  **Check System Instructions:**
    *   **Role:** Useful AI assistant, helping users answer questions and complete tasks.
    *   **Tone:** Direct, clear, restrained. No exaggeration, emojis, or long preambles unless necessary.
    *   **Output:** Prioritize action/conclusion, then explanation. Mainly serve three things: sync progress, explain blocks, give final result.
    *   **Context:** I am an AI assistant. I should not pretend to be something else.
    *   **Safety:** Avoid security issues, etc. (Not relevant for a self-intro, but good to keep in mind).

3.  **Drafting the Response:**
 

In [8]:
agent.with_skill(CalculatorSkill())
agent.with_skill(TranslateSkill())

2026-04-19 01:58:28,954 | INFO | 📦 注册 Skill 'calculator' (v1.0.0)
2026-04-19 01:58:28,955 | INFO | ✅ 激活 Skill 'calculator' (工具: ['calculator'])
2026-04-19 01:58:28,956 | INFO | 📦 注册 Skill 'translate' (v1.0.0)
2026-04-19 01:58:28,956 | INFO | ✅ 激活 Skill 'translate' (工具: ['translate_tool'])


In [9]:
test_invoke_with_tool(agent)

2026-04-19 01:58:34,330 | INFO | 对话历史已清空
2026-04-19 01:58:34,331 | INFO | 使用工具模式调用智能体
2026-04-19 01:58:36,790 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-19 01:58:36,792 | INFO | 思考内容: 用户要求我：
1. 使用工具翻译"你是谁，在哪里"到英语
2. 判断这个翻译工具是否正确
3. 计算 3^22

我需要：
1. 调用 translate_tool 翻译中文到英文
2. 调用 calculator 计算 3^22

这两个任务都是独立的，可以并行调用工具。

2026-04-19 01:58:36,794 | INFO | test_skill执行工具: translate_tool，参数: {'text': '你是谁，在哪里', 'target_lang': 'en'}
2026-04-19 01:58:36,795 | INFO | test_skill执行工具: calculator，参数: {'expression': '3^22'}
2026-04-19 01:58:38,383 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-19 01:58:38,385 | INFO | 思考内容: 我需要检查翻译工具的结果。看起来翻译工具返回的只是原文，没有翻译成英语。这是一个问题。

让我重新翻译，看看是否能得到正确的结果。

2026-04-19 01:58:38,385 | INFO | test_skill执行工具: translate_tool，参数: {'text': '你是谁，在哪里', 'target_lang': 'en'}
2026-04-19 01:58:41,004 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1



翻译工具**不正确**，计算器**正确**。

### 结果：

1. **翻译工具** ❌
   - 输入：你是谁，在哪里
   - 返回：你是谁，在哪里（未翻译）
   - 工具未能将中文正确翻译为英语

2. **计算器** ✅
   - 计算：3^22 = **31,381,059,609**
   - 结果正确


In [ ]:
await test_ainvoke_with_tool(agent)

In [ ]:
test_stream_with_tool(agent)

In [10]:
await test_astream_with_tool(agent)

2026-04-19 01:58:45,405 | INFO | 对话历史已清空
2026-04-19 01:58:45,429 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"


round 1

thinking content:
用户要求我：
1. 翻译中文文字到英语
2. 计算 3^22

我需要：
1. 使用 translate_tool 翻译 "你是谁，在哪里" 到英语
2. 使用 calculator 计算 3^22

这两个任务可以并行执行，因为它们是独立的。

让我先调用这两个工具。

content:



tool_calls:
translate_tool : {'text': '你是谁，在哪里', 'target_lang': 'en'}
calculator : {'expression': '3^22'}


2026-04-19 01:58:47,676 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"



round 2

thinking content:
翻译工具返回的结果似乎有问题，它把中文原样返回了，没有正确翻译成英语。计算器工具的结果是正确的：3^22 = 31381059609。

让我重新尝试翻译。

content:



tool_calls:
translate_tool : {'text': '你是谁，你在哪里', 'target_lang': 'en'}


2026-04-19 01:58:49,264 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"



round 3

thinking content:
用户要求：
1. 翻译"你是谁，在哪里"到英语
2. 计算 3^22

第一个工具调用返回的翻译是"你是谁，你在哪里"，这看起来是中文，不是英语。这说明翻译工具可能出错了，或者返回了错误的结果。

第二个工具调用计算了 3^22 = 31381059609

让我重新调用翻译工具，并验证结果是否正确。

实际上，我注意到第一次翻译返回的结果仍然是中文，这说明翻译工具可能有问题。让我尝试用更明确的请求重新翻译。

content:


翻译工具似乎没有正确工作，返回的结果仍然是中文而不是英语。让我重新尝试翻译，并验证计算结果：


tool_calls:
translate_tool : {'text': '你是谁，你在哪里', 'target_lang': 'en'}


2026-04-19 01:58:52,123 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"



round 4

thinking content:
翻译工具似乎有问题，它没有正确地将中文翻译成英文，而是返回了相同的中文文本。计算器的结果是 3^22 = 31381059609。

我应该如实告知用户翻译工具工作不正常，并给出计算结果。

content:


翻译工具未能正常工作，返回的仍是中文原文而非英文翻译。

**计算结果：**
3²² = 31,381,059,609

**翻译问题：**
工具返回了与输入相同的中文文本，未能完成翻译。这可能是因为：
- 源语言识别问题
- 目标语言参数未生效
- 工具本身存在故障

如需准确翻译，建议手动处理或使用其他翻译服务。
final res:


翻译工具未能正常工作，返回的仍是中文原文而非英文翻译。

**计算结果：**
3²² = 31,381,059,609

**翻译问题：**
工具返回了与输入相同的中文文本，未能完成翻译。这可能是因为：
- 源语言识别问题
- 目标语言参数未生效
- 工具本身存在故障

如需准确翻译，建议手动处理或使用其他翻译服务。


In [ ]:
raw_history=agent.get_raw_history()

In [11]:
llm= EasyLLM(provider="anthropic_native",base_url="http://127.0.0.1:5124",api_key="122",model="qwen3.5-9b")
agent.change_model(llm=llm)

2026-04-19 01:58:56,658 | INFO | EasyLLM 初始化完成: provider=anthropic_native, model=qwen3.5-9b


In [ ]:
raw_history2=agent.get_raw_history()  

In [ ]:
raw_history==raw_history2

In [12]:
await agent.astream_invoke(f"在帮我翻译一下:我是一直小白马")


2026-04-19 01:59:02,456 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/messages "HTTP/1.1 200 OK"


round 1

thinking content:
用户要求我翻译"我是一直小白马"这句话，但根据之前的经验，翻译工具似乎工作不正常。让我再尝试一次，看看是否这次能正常工作。

不过我也注意到用户写的是"一直"应该是"一只"（量词用错了），我应该指出这一点。

让我先调用翻译工具看看结果。

content:



tool_calls:
translate_tool : {'text': '我是一只小白马', 'target_lang': 'en'}


2026-04-19 01:59:04,352 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/messages "HTTP/1.1 200 OK"



round 2

thinking content:
翻译工具仍然没有正常工作，返回的仍然是中文原文而不是英文翻译。我需要向用户说明这个工具的问题，并尝试提供正确的翻译。

"我是一只小白马"翻译成英语应该是 "I am a little white horse" 或 "I am a small white horse"。

我应该告诉用户翻译工具存在问题，并提供我手动翻译的结果。

content:


翻译工具仍然无法正常工作，返回的是原文而非英文翻译。

**手动翻译结果：**
- "我是一只小白马" → "I am a little white horse"

**说明：**
翻译工具似乎存在缺陷，多次调用均返回原文。建议：
1. 如需准确翻译，可使用其他翻译工具（如 Google 翻译、百度翻译等）
2. 或者手动查阅词典获取翻译

如需计算其他内容或其他帮助，请随时告诉我。
final res:


翻译工具仍然无法正常工作，返回的是原文而非英文翻译。

**手动翻译结果：**
- "我是一只小白马" → "I am a little white horse"

**说明：**
翻译工具似乎存在缺陷，多次调用均返回原文。建议：
1. 如需准确翻译，可使用其他翻译工具（如 Google 翻译、百度翻译等）
2. 或者手动查阅词典获取翻译

如需计算其他内容或其他帮助，请随时告诉我。


/home/wxd/miniconda3/envs/llm/lib/python3.10/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ParsedTextBlock[~ResponseFormatT]` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock(citations...xt', parsed_output=None), input_type=ParsedTextBlock])
  PydanticSerializationUnexpectedValue(Expected `ThinkingBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock(citations...xt', parsed_output=None), input_type=ParsedTextBlock])
  PydanticSerializationUnexpectedValue(Expected `RedactedThinkingBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock(citations...xt', parsed_output=None), input_type=ParsedTextBlock])
  PydanticSerializationUnexpectedValue(Expected `ToolUseBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock(citations...xt', par

'\n\n翻译工具仍然无法正常工作，返回的是原文而非英文翻译。\n\n**手动翻译结果：**\n- "我是一只小白马" → "I am a little white horse"\n\n**说明：**\n翻译工具似乎存在缺陷，多次调用均返回原文。建议：\n1. 如需准确翻译，可使用其他翻译工具（如 Google 翻译、百度翻译等）\n2. 或者手动查阅词典获取翻译\n\n如需计算其他内容或其他帮助，请随时告诉我。'

In [13]:
llm= EasyLLM(provider="openai",base_url="http://127.0.0.1:5124/v1",api_key="122",model="qwen3.5-9b")
agent.change_model(llm=llm)

2026-04-19 01:59:25,785 | INFO | EasyLLM 初始化完成: provider=openai, model=qwen3.5-9b


In [14]:
raw_history3=agent.get_raw_history()  
raw_history==raw_history3

NameError: name 'raw_history' is not defined

In [15]:
await agent.astream_invoke(f"我们刚才聊了什么")


2026-04-19 01:59:30,863 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"


round 1

thinking content:
用户问我们刚才聊了什么，我需要回顾一下对话历史：

1. 用户最初要求：
   - 翻译"你是谁，在哪里"到英语
   - 计算 3^22

2. 我尝试使用翻译工具，但发现工具返回了原文而不是英文翻译

3. 我手动计算了 3^22 = 31,381,059,609

4. 用户又让我翻译"我是一直小白马"

5. 翻译工具再次失败，返回原文

6. 我手动提供了翻译："I am a little white horse"

7. 现在用户问我刚才聊了什么

我应该简洁地总结对话内容，不需要调用任何工具。

content:


我们刚才聊了以下内容：

1. **翻译请求**：你让我将"你是谁，在哪里"翻译成英语，但翻译工具未能正常工作，返回了原文
2. **数学计算**：我计算了 3²² = 31,381,059,609
3. **再次翻译**：你让我翻译"我是一只小白马"，翻译工具同样未能正确翻译
4. **手动翻译**：我提供了该句的英文翻译："I am a little white horse"

总结：主要讨论了翻译工具的问题，并完成了数学计算任务。
final res:


我们刚才聊了以下内容：

1. **翻译请求**：你让我将"你是谁，在哪里"翻译成英语，但翻译工具未能正常工作，返回了原文
2. **数学计算**：我计算了 3²² = 31,381,059,609
3. **再次翻译**：你让我翻译"我是一只小白马"，翻译工具同样未能正确翻译
4. **手动翻译**：我提供了该句的英文翻译："I am a little white horse"

总结：主要讨论了翻译工具的问题，并完成了数学计算任务。


'\n\n我们刚才聊了以下内容：\n\n1. **翻译请求**：你让我将"你是谁，在哪里"翻译成英语，但翻译工具未能正常工作，返回了原文\n2. **数学计算**：我计算了 3²² = 31,381,059,609\n3. **再次翻译**：你让我翻译"我是一只小白马"，翻译工具同样未能正确翻译\n4. **手动翻译**：我提供了该句的英文翻译："I am a little white horse"\n\n总结：主要讨论了翻译工具的问题，并完成了数学计算任务。'